# 9 WorkFlow Analista Jr - Feature Engineering Intra-mes (Grupo A) & Multi-Semilla


### 9.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
<br>El Analista Jr corre sus scripts en la virtual manchine **desktop-jr** que tiene estas características


*   Normal, paga tarifa completa, nunca es apagada por Google
*   reside en el datacenter de Toronto, Canada
*   64 GB de memoria RAM
*   8 vCPU


En Analista Jr **no** puede utilizar Google Colab porque los 12 GB de dichas maquinas virtuales no son suficientes para el tamaño del dataset que está utilizando.



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Fri Sep 11 16:17:12 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671174,35.9,1479542,79.1,1479542,79.1
Vcells,1242565,9.5,8388608,64.0,1978697,15.1


In [3]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, u

#### Parametros

In [4]:
PARAM <- list()
# Vector de semillas a iterar (soporta una o varias semillas)
PARAM$semillas <- c(100313,	214351,	357913,	701819,	900001)
PARAM$semilla_primigenia <- PARAM$semillas[1] # Semilla base / retrocompatibilidad

# Estrategia de optimización de hiperparámetros en multi-semilla:
# FALSE: Optimiza hiperparámetros una sola vez con la primera semilla y reutiliza los mejores para todas las semillas (Ahorra ~65 min por semilla extra).
# TRUE:  Ejecuta Grid Search completo de forma independiente para cada una de las semillas.
PARAM$gridsearch_por_semilla <- FALSE

# Configuración del Algoritmo Genético de Feature Engineering Intra-mes (gramEvol)
PARAM$GA <- list(
  popSize = 200,            # Tamaño de la población de individuos
  iterations = 50,          # Cantidad de generaciones evolutivas
  top_features = 10,         # Cantidad de mejores features no lineales a inyectar al dataset
  max_terminales = 60,      # Máximo de variables numéricas base a combinar
  seqLen = 200,             # Longitud máxima de codones del genoma
  max.depth = 10,           # Profundidad máxima del árbol sintáctico BNF
  max_filas_fitness = 50000 # Muestra máxima para evaluación de fitness ultrarrápida

)

PARAM$experimento <- 9111
PARAM$dataset <- "analistajr_competencia_2026.csv.gz"


#### Carpeta del Experimento

In [5]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
dir_experimento_base <- paste0("/content/buckets/b1/exp/", experimento_folder)
setwd( dir_experimento_base )


### 9.3.1   Preprocesamiento del dataset

#### 9.3.1.1  DT incorporar dataset

In [6]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [7]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

Loading required package: mice


Attaching package: ‘mice’


The following object is masked from ‘package:stats’:

    filter


The following objects are masked from ‘package:base’:

    cbind, rbind




In [8]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}


In [9]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(envg$PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(envg$PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [10]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [11]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [12]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [13]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

inicio Corregir_Rotas()
fin Corregir_rotas()


#### 9.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [14]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [15]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [16]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

IPC,dolar_blue,dolar_oficial,UVA,foto_mes
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1.9903031,39.04545,38.43000,2.0014088,201901
1.9174404,38.40250,39.42800,1.9503255,201902
1.8296187,41.63947,42.54210,1.8932303,201903
1.7728863,44.27474,44.35421,1.8247220,201904
1.7212488,46.09546,46.08864,1.7460278,201905
1.6776304,45.06333,44.95500,1.6871348,201906
1.6431248,43.98333,43.75143,1.6361679,201907
1.5814483,54.84286,54.65048,1.5927530,201908
1.4947527,61.05952,58.79000,1.5549163,201909


In [17]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [18]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [19]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [20]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [21]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [22]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [23]:
drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [24]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

[1] "mrentabilidad"                      "mrentabilidad_annual"              
 [3] "mcomisiones"                        "mactivos_margen"                   
 [5] "mpasivos_margen"                    "mcuenta_corriente"                 
 [7] "mcaja_ahorro"                       "mcuentas_saldo"                    
 [9] "mtarjeta_visa_consumo"              "mtarjeta_master_consumo"           
[11] "mprestamos_personales"              "mpayroll"                          
[13] "mttarjeta_visa_debitos_automaticos" "mcomisiones_mantenimiento"         
[15] "mtransferencias_recibidas"          "Master_mfinanciacion_limite"       
[17] "Master_msaldototal"                 "Master_mlimitecompra"              
[19] "Master_mconsumototal"               "Master_mpagominimo"                
[21] "Visa_mfinanciacion_limite"          "Visa_msaldototal"                  
[23] "Visa_mlimitecompra"                 "Visa_mconsumototal"                
[25] "Visa_mpagominimo"

In [25]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "deflacion"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


inicio drift_deflacion()
fin drift_deflacion()


In [26]:
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"

In [27]:
# se intenta corregir el data drifting utilizando algunos indices financieros

#### 9.3.1.3.1 FE_intra_manual: Feature Engineering intra-mes manual (Reglas de Negocio)


In [28]:
# ==============================================================================
# 9.3.1.3 FE_intra_manual Feature Engineering intra-mes
# ==============================================================================

# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# --- Variables Baseline ---
# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / (cliente_edad + 1)]

# ==============================================================================
# INICIO NUEVO BLOQUE DE FEATURE ENGINEERING INTRA-MES (FAMILIAS A - F)
# Se asegura agregar siempre un termino amortiguador (+ 1) en cada denominador
# para evitar divisiones por cero o valores infinitos.
# ==============================================================================

# ------------------------------------------------------------------------------
# A. Utilización y Estrés de Tarjetas de Crédito
# ------------------------------------------------------------------------------
# Visa_utilizacion = Visa_msaldototal / (Visa_mlimitecompra + 1)
if( atributos_presentes( c("Visa_msaldototal", "Visa_mlimitecompra") ) )
  dataset[, Visa_utilizacion := Visa_msaldototal / (Visa_mlimitecompra + 1)]

# Master_utilizacion = Master_msaldototal / (Master_mlimitecompra + 1)
if( atributos_presentes( c("Master_msaldototal", "Master_mlimitecompra") ) )
  dataset[, Master_utilizacion := Master_msaldototal / (Master_mlimitecompra + 1)]

# Visa_estres_fin = Visa_msaldototal / (Visa_mfinanciacion_limite + 1)
if( atributos_presentes( c("Visa_msaldototal", "Visa_mfinanciacion_limite") ) )
  dataset[, Visa_estres_fin := Visa_msaldototal / (Visa_mfinanciacion_limite + 1)]

# Master_estres_fin = Master_msaldototal / (Master_mfinanciacion_limite + 1)
if( atributos_presentes( c("Master_msaldototal", "Master_mfinanciacion_limite") ) )
  dataset[, Master_estres_fin := Master_msaldototal / (Master_mfinanciacion_limite + 1)]

# Visa_pago_minimo_ratio = Visa_mpagominimo / (Visa_msaldototal + 1)
if( atributos_presentes( c("Visa_mpagominimo", "Visa_msaldototal") ) )
  dataset[, Visa_pago_minimo_ratio := Visa_mpagominimo / (Visa_msaldototal + 1)]

# Master_pago_minimo_ratio = Master_mpagominimo / (Master_msaldototal + 1)
if( atributos_presentes( c("Master_mpagominimo", "Master_msaldototal") ) )
  dataset[, Master_pago_minimo_ratio := Master_mpagominimo / (Master_msaldototal + 1)]

# ------------------------------------------------------------------------------
# B. Consolidación de Tarjetas (Visa + Mastercard)
# ------------------------------------------------------------------------------
# TC_deuda_total = Visa_msaldototal + Master_msaldototal
if( atributos_presentes( c("Visa_msaldototal", "Master_msaldototal") ) )
  dataset[, TC_deuda_total := Visa_msaldototal + Master_msaldototal]

# TC_consumo_total = mtarjeta_visa_consumo + mtarjeta_master_consumo
if( atributos_presentes( c("mtarjeta_visa_consumo", "mtarjeta_master_consumo") ) )
  dataset[, TC_consumo_total := mtarjeta_visa_consumo + mtarjeta_master_consumo]

# TC_trx_total = ctarjeta_visa_transacciones + ctarjeta_master_transacciones
if( atributos_presentes( c("ctarjeta_visa_transacciones", "ctarjeta_master_transacciones") ) )
  dataset[, TC_trx_total := ctarjeta_visa_transacciones + ctarjeta_master_transacciones]

# TC_limite_total = Visa_mlimitecompra + Master_mlimitecompra
if( atributos_presentes( c("Visa_mlimitecompra", "Master_mlimitecompra") ) )
  dataset[, TC_limite_total := Visa_mlimitecompra + Master_mlimitecompra]

# ------------------------------------------------------------------------------
# C. Cobertura de Deuda e Ingresos (Payroll)
# ------------------------------------------------------------------------------
# ratio_deuda_sueldo = (TC_deuda_total + mprestamos_personales) / (mpayroll + 1)
if( atributos_presentes( c("TC_deuda_total", "mprestamos_personales", "mpayroll") ) )
  dataset[, ratio_deuda_sueldo := (TC_deuda_total + mprestamos_personales) / (mpayroll + 1)]

# ratio_liquidez_sueldo = mcuentas_saldo / (mpayroll + 1)
if( atributos_presentes( c("mcuentas_saldo", "mpayroll") ) )
  dataset[, ratio_liquidez_sueldo := mcuentas_saldo / (mpayroll + 1)]

# ratio_consumo_sueldo = TC_consumo_total / (mpayroll + 1)
if( atributos_presentes( c("TC_consumo_total", "mpayroll") ) )
  dataset[, ratio_consumo_sueldo := TC_consumo_total / (mpayroll + 1)]

# ------------------------------------------------------------------------------
# D. Ratios de Ticket Promedio
# ------------------------------------------------------------------------------
# Visa_ticket_prom = mtarjeta_visa_consumo / (ctarjeta_visa_transacciones + 1)
if( atributos_presentes( c("mtarjeta_visa_consumo", "ctarjeta_visa_transacciones") ) )
  dataset[, Visa_ticket_prom := mtarjeta_visa_consumo / (ctarjeta_visa_transacciones + 1)]

# Master_ticket_prom = mtarjeta_master_consumo / (ctarjeta_master_transacciones + 1)
if( atributos_presentes( c("mtarjeta_master_consumo", "ctarjeta_master_transacciones") ) )
  dataset[, Master_ticket_prom := mtarjeta_master_consumo / (ctarjeta_master_transacciones + 1)]

# TC_ticket_prom_total = TC_consumo_total / (TC_trx_total + 1)
if( atributos_presentes( c("TC_consumo_total", "TC_trx_total") ) )
  dataset[, TC_ticket_prom_total := TC_consumo_total / (TC_trx_total + 1)]

# ------------------------------------------------------------------------------
# E. Digitalización, Rentabilidad y Antigüedad
# ------------------------------------------------------------------------------
# ratio_hb_trx = chomebanking_transacciones / ((ctrx_quarter / 3) + 1)
if( atributos_presentes( c("chomebanking_transacciones", "ctrx_quarter") ) )
  dataset[, ratio_hb_trx := chomebanking_transacciones / ((ctrx_quarter / 3) + 1)]

# ratio_rentabilidad_anual = mrentabilidad / ((mrentabilidad_annual / 12) + 1)
if( atributos_presentes( c("mrentabilidad", "mrentabilidad_annual") ) )
  dataset[, ratio_rentabilidad_anual := mrentabilidad / ((mrentabilidad_annual / 12) + 1)]

# velocidad_productos = cproductos / (cliente_antiguedad + 1)
if( atributos_presentes( c("cproductos", "cliente_antiguedad") ) )
  dataset[, velocidad_productos := cproductos / (cliente_antiguedad + 1)]

# ------------------------------------------------------------------------------
# F. Flags e Indicadores Binarios
# ------------------------------------------------------------------------------
# flag_sobregiro = as.integer(mcuenta_corriente < 0 & cdescubierto_preacordado == 1)
if( atributos_presentes( c("mcuenta_corriente", "cdescubierto_preacordado") ) )
  dataset[, flag_sobregiro := as.integer(mcuenta_corriente < 0 & cdescubierto_preacordado == 1)]

# flag_inactivo_deuda = as.integer(ctrx_quarter == 0 & TC_deuda_total > 0)
if( atributos_presentes( c("ctrx_quarter", "TC_deuda_total") ) )
  dataset[, flag_inactivo_deuda := as.integer(ctrx_quarter == 0 & TC_deuda_total > 0)]

# flag_doble_tarjeta = as.integer(ctarjeta_visa > 0 & ctarjeta_master > 0)
if( atributos_presentes( c("ctarjeta_visa", "ctarjeta_master") ) )
  dataset[, flag_doble_tarjeta := as.integer(ctarjeta_visa > 0 & ctarjeta_master > 0)]

# ==============================================================================
# FIN NUEVO BLOQUE DE FEATURE ENGINEERING INTRA-MES
# ==============================================================================


#### 9.3.1.3.2 FE_intra_GA: Feature Engineering Intra-mes con Algoritmo Genético (Grammatical Evolution - gramEvol)
##### Problema #10: Búsqueda heurística evolutiva de combinaciones no lineales
A continuación se implementa la evolución gramatical mediante el paquete `gramEvol`:
1. **Gramática formal BNF (Backus-Naur Form)**: Define el espacio de búsqueda algebraico permitiendo operadores aritméticos protegidos (`+`, `-`, `*`, `protected_div`, `protected_log_diff`) y un pool de variables numéricas candidatas.
2. **Función de Fitness con LightGBM Univariado Real**: Cada individuo generado en la población es evaluado mediante el AUC de validación de un modelo LightGBM univariado entrenado sobre un split local (sin tocar el mes `202107` de validación oficial).
3. **Mapeo Genotipo a Fenotipo**: Conversión unívoca del genoma a expresión R evaluable.
4. **Extracción e Inyección del Top N**: Se seleccionan las mejores 5 fórmulas evolutivas no triviales ni duplicadas, se evalúan sobre todo el dataset y se inyectan como columnas `GA_Feature_1`, `GA_Feature_2`, ..., `GA_Feature_5`.


In [29]:
# ==============================================================================
# 9.3.1.3.2 FE_intra_GA: Algoritmo Genético (Grammatical Evolution)
# ==============================================================================

if (!require("gramEvol")) {
  install.packages("gramEvol", repos = "https://cloud.r-project.org", dependencies = TRUE)
}
if (!require("gramEvol")) {
  install.packages("gramEvol", repos = "https://cran.rstudio.com", dependencies = TRUE)
}
require("gramEvol")
require("lightgbm")
require("data.table")
require("parallel")

if (is.null(PARAM$GA)) {
  PARAM$GA <- list(
    popSize = 100,
    iterations = 30,
    top_features = 5,
    max_terminales = 40,
    seqLen = 200,
    max.depth = 10,
    max_filas_fitness = 50000
  )
}

if (!"clase01" %in% colnames(dataset)) {
  dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1, 0)]
}

# 1. Variables candidatas para el Algoritmo Genético
excluir_GA <- c("numero_de_cliente", "foto_mes", "clase_ternaria", "clase01", "azar", "fold_train", "fold_final_train")
candidatas_GA <- setdiff(colnames(dataset), excluir_GA)

# Filtrar únicamente variables numéricas
son_numericas <- sapply(dataset[, candidatas_GA, with = FALSE], is.numeric)
candidatas_GA <- candidatas_GA[son_numericas]

# Excluir fechas
patron_fechas <- "^(f|.*_f|.*fecha)"
candidatas_GA <- candidatas_GA[!grepl(patron_fechas, candidatas_GA, ignore.case = TRUE)]

# Límite de seguridad de terminales para controlar el espacio de búsqueda
MAX_TERMINALES <- if (!is.null(PARAM$GA$max_terminales)) PARAM$GA$max_terminales else 40
if (length(candidatas_GA) > MAX_TERMINALES) {
  set.seed(PARAM$semilla_primigenia)
  candidatas_GA <- sample(candidatas_GA, MAX_TERMINALES)
}

cat("Variables candidatas para Grammatical Evolution:", length(candidatas_GA), "\n")

# 2. Split Local de Validación (sin tocar 202107 ni 202109)
meses_disponibles <- sort(unique(dataset$foto_mes[dataset$foto_mes < 202107]))
meses_val_GA <- tail(meses_disponibles, 3)
meses_tr_GA  <- setdiff(meses_disponibles, meses_val_GA)

idx_tr_all <- which(dataset$foto_mes %in% meses_tr_GA)
idx_val_all <- which(dataset$foto_mes %in% meses_val_GA)

# Subsampling controlado para evaluación de fitness ultrarrápida
set.seed(PARAM$semilla_primigenia)
n_fit <- if (!is.null(PARAM$GA$max_filas_fitness)) PARAM$GA$max_filas_fitness else 50000
idx_tr_GA <- if (length(idx_tr_all) > n_fit) sample(idx_tr_all, n_fit) else idx_tr_all
idx_val_GA <- if (length(idx_val_all) > (n_fit / 2)) sample(idx_val_all, n_fit / 2) else idx_val_all

y_tr_GA <- dataset$clase01[idx_tr_GA]
y_val_GA <- dataset$clase01[idx_val_GA]

# 3. Operadores Protegidos
protected_div <- function(x, y) {
  res <- x / (y + 1e-5)
  res[is.na(res) | is.infinite(res)] <- 0
  res
}

protected_log_diff <- function(x, y) {
  res <- log(abs(x - y) + 1)
  res[is.na(res) | is.infinite(res)] <- 0
  res
}

es_expresion_trivial <- function(f) {
  !grepl("[+*/-]|protected_", f)
}

# 4. Definición de la Gramática BNF
string_vars <- paste(candidatas_GA, collapse = " | ")
rule_text <- paste0(
  "<expr> ::= <op>\n",
  "<op>   ::= <op> + <op> | <op> - <op> | <op> * <op> | ",
  "protected_div(<op>, <op>) | protected_log_diff(<op>, <op>) | <var>\n",
  "<var>  ::= ", string_vars
)

tf <- tempfile()
writeLines(rule_text, tf)
bnf_grammar <- CreateGrammar(tf)
unlink(tf)

# 5. Función de Fitness con LightGBM Univariado Real
fitness_gramEvol <- function(expr) {
  valores <- tryCatch(eval(expr, envir = dataset), error = function(e) NULL)
  if (is.null(valores)) return(1)

  val_tr <- valores[idx_tr_GA]
  val_finitos <- val_tr[is.finite(val_tr)]
  if (length(val_finitos) == 0 || length(unique(val_finitos)) <= 1) {
    return(1)
  }

  dtr_ga  <- lgb.Dataset(data = matrix(val_tr, ncol = 1), label = y_tr_GA, free_raw_data = TRUE)
  dval_ga <- lgb.Dataset(data = matrix(valores[idx_val_GA], ncol = 1), label = y_val_GA, free_raw_data = TRUE)

  modelo_ga <- tryCatch({
    lgb.train(
      params = list(objective = "binary", metric = "auc",
                    learning_rate = 0.1, num_threads = 1, verbosity = -1),
      data = dtr_ga, valids = list(valid = dval_ga),
      nrounds = 50, early_stopping_rounds = 10, verbose = -1
    )
  }, error = function(e) NULL)

  if (is.null(modelo_ga) || is.null(modelo_ga$best_score) || is.na(modelo_ga$best_score)) return(1)

  1 - modelo_ga$best_score
}

evaluar_genoma <- function(genoma) {
  expr_obj <- tryCatch(suppressWarnings(GrammarMap(genoma, bnf_grammar)), error = function(e) NULL)
  if (is.null(expr_obj) || !isTRUE(GrammarIsTerminal(expr_obj))) {
    return(list(score = 0, formula = NA_character_))
  }
  expr_lang <- tryCatch(as.expression(expr_obj), error = function(e) NULL)
  if (is.null(expr_lang) || length(expr_lang) == 0) {
    return(list(score = 0, formula = NA_character_))
  }

  expr_final  <- expr_lang[[1]]
  formula_str <- paste(deparse(expr_final, width.cutoff = 500L), collapse = " ")
  costo <- fitness_gramEvol(expr_final)
  auc   <- 1 - costo

  list(score = auc, formula = formula_str)
}

# 6. Ejecución del Algoritmo Genético
set.seed(PARAM$semilla_primigenia)

cat("\n===================================================================\n")
cat(">>> INICIANDO GRAMMATICAL EVOLUTION (gramEvol) <<<\n")
cat("===================================================================\n")

ge_res <- GrammaticalEvolution(
  grammarDef      = bnf_grammar,
  evalFunc        = fitness_gramEvol,
  popSize         = PARAM$GA$popSize,
  iterations      = PARAM$GA$iterations,
  terminationCost = 0.10,
    mutationChance	= 0.20,
  seqLen          = PARAM$GA$seqLen,
  max.depth       = PARAM$GA$max.depth,
  monitorFunc     = function(result) {
    cat(sprintf("Gen %2d | Mejor Costo: %.5f (AUC: %.5f)\n",
                result$population$currentIteration,
                result$best$cost,
                1 - result$best$cost))
  }
)

# 7. Extracción e Inyección del Top N al Dataset
cat("\n=== Evaluando población final para extraer el Top", PARAM$GA$top_features, "===\n")

pop_matrix      <- ge_res$population$population
poblacion_final <- split(pop_matrix, row(pop_matrix))

n_cores <- max(1, detectCores() - 1)
resultados <- if (.Platform$OS.type == "unix") {
  mclapply(poblacion_final, evaluar_genoma, mc.cores = n_cores)
} else {
  lapply(poblacion_final, evaluar_genoma)
}

scores_finales   <- sapply(resultados, function(r) r$score)
formulas_finales <- sapply(resultados, function(r) r$formula)

ordenados <- order(scores_finales, decreasing = TRUE)

formulas_vistas <- character()
ga_cols_creadas <- character()
top_guardados   <- 0
idx             <- 1

while (top_guardados < PARAM$GA$top_features && idx <= length(ordenados)) {
  i <- ordenados[idx]
  idx <- idx + 1

  if (is.na(scores_finales[i]) || scores_finales[i] <= 0.50) next
  if (is.na(formulas_finales[i]) || formulas_finales[i] %in% formulas_vistas) next
  if (es_expresion_trivial(formulas_finales[i])) next

  eval_res <- tryCatch(
    eval(parse(text = formulas_finales[i])[[1]], envir = dataset),
    error = function(e) NULL
  )
  if (is.null(eval_res) || length(unique(eval_res[is.finite(eval_res)])) <= 1) next

  top_guardados <- top_guardados + 1
  formulas_vistas <- c(formulas_vistas, formulas_finales[i])

  nombre_col <- paste0("GA_Feature_", top_guardados)
  ga_cols_creadas <- c(ga_cols_creadas, nombre_col)

  cat(sprintf("[%s] AUC Univariado: %.5f | Fórmula: %s\n", nombre_col, scores_finales[i], formulas_finales[i]))
  dataset[, (nombre_col) := eval_res]
}

cat("\nColumnas generadas por Algoritmo Genético e inyectadas al dataset:", paste(ga_cols_creadas, collapse = ", "), "\n")


Loading required package: gramEvol

Loading required package: lightgbm

Loading required package: parallel



Variables candidatas para Grammatical Evolution: 60 

>>> INICIANDO GRAMMATICAL EVOLUTION (gramEvol) <<<
Gen  1 | Mejor Costo: 0.15808 (AUC: 0.84192)
Gen  2 | Mejor Costo: 0.15234 (AUC: 0.84766)
Gen  3 | Mejor Costo: 0.15234 (AUC: 0.84766)
Gen  4 | Mejor Costo: 0.15234 (AUC: 0.84766)
Gen  5 | Mejor Costo: 0.15234 (AUC: 0.84766)
Gen  6 | Mejor Costo: 0.15234 (AUC: 0.84766)
Gen  7 | Mejor Costo: 0.14874 (AUC: 0.85126)
Gen  8 | Mejor Costo: 0.14874 (AUC: 0.85126)
Gen  9 | Mejor Costo: 0.14874 (AUC: 0.85126)
Gen 10 | Mejor Costo: 0.14874 (AUC: 0.85126)
Gen 11 | Mejor Costo: 0.14874 (AUC: 0.85126)
Gen 12 | Mejor Costo: 0.14874 (AUC: 0.85126)
Gen 13 | Mejor Costo: 0.14874 (AUC: 0.85126)
Gen 14 | Mejor Costo: 0.14874 (AUC: 0.85126)
Gen 15 | Mejor Costo: 0.14874 (AUC: 0.85126)
Gen 16 | Mejor Costo: 0.14874 (AUC: 0.85126)
Gen 17 | Mejor Costo: 0.14874 (AUC: 0.85126)
Gen 18 | Mejor Costo: 0.14874 (AUC: 0.85126)
Gen 19 | Mejor Costo: 0.14874 (AUC: 0.85126)
Gen 20 | Mejor Costo: 0.14874 (AUC: 0.85

In [30]:
# Auditoría de resultados del Algoritmo Genético e inspección de columnas
cat("Total individuos en población final:", length(scores_finales), "\n")
cat("Individuos con score > 0.50:", sum(scores_finales > 0.50, na.rm = TRUE), "\n")
cat("Expresiones no triviales con score > 0.50:", sum(scores_finales > 0.50 & !sapply(formulas_finales, es_expresion_trivial), na.rm = TRUE), "\n")
cat("Expresiones únicas no triviales:", length(unique(formulas_finales[scores_finales > 0.50 & !sapply(formulas_finales, es_expresion_trivial)])), "\n")
cat("\nColumnas actuales del dataset tras Feature Engineering Intra-mes (Manual + GA):\n")
colnames(dataset)


Total individuos en población final: 201 
Individuos con score > 0.50: 23 
Expresiones no triviales con score > 0.50: 7 
Expresiones únicas no triviales: 7 

Columnas actuales del dataset tras Feature Engineering Intra-mes (Manual + GA):


[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"                     "kmes"                              
[57] "mpayroll_sobre_edad"                "Visa_utilizacion"                  
[59] "Master_utilizacion"                 "Visa_estres_fin"                   
[61] "Master_estres_fin"                  "Visa_pago_minimo_ratio"            
[63] "Master_pago_minimo_ratio"           "TC_deuda_total"                    
[65] "TC_consumo_total"                   "TC_trx_total"                      
[67] "TC_limite_total"                    "ratio_deuda_sueldo"                
[69] "ratio_liquidez_sueldo"              "ratio_consumo_sueldo"              
[71] "Visa_ticket_prom"                   "Master_ticket_prom"                
[73] "TC_ticket_prom_total"               "ratio_hb_trx"                      
[75] "ratio_rentabilidad_anual"           "velocidad_productos"               
[77] "flag_sobregiro"                     "flag_inactivo_deuda"               
[79] "flag_doble_tarjeta"                 "clase01"                           
[81] "GA_Feature_1"                       "GA_Feature_2"                      
[83] "GA_Feature_3"                       "GA_Feature_4"                      
[85] "GA_Feature_5"                       "GA_Feature_6"                      
[87] "GA_Feature_7"

#### 9.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

In [31]:
# No se implementa Feature Engineering a partir de Random Forest

#### 9.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [32]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [33]:
ncol(dataset)
colnames(dataset)

[1] 423

[1] "numero_de_cliente"                        
  [2] "foto_mes"                                 
  [3] "internet"                                 
  [4] "cliente_edad"                             
  [5] "cliente_antiguedad"                       
  [6] "mrentabilidad"                            
  [7] "mrentabilidad_annual"                     
  [8] "mcomisiones"                              
  [9] "mactivos_margen"                          
 [10] "mpasivos_margen"                          
 [11] "cproductos"                               
 [12] "mcuenta_corriente"                        
 [13] "mcaja_ahorro"                             
 [14] "cdescubierto_preacordado"                 
 [15] "mcuentas_saldo"                           
 [16] "ctarjeta_visa"                            
 [17] "ctarjeta_visa_transacciones"              
 [18] "mtarjeta_visa_consumo"                    
 [19] "ctarjeta_master"                          
 [20] "ctarjeta_master_transacciones"            
 [21] "mtarjeta_master_consumo"                  
 [22] "cprestamos_personales"                    
 [23] "mprestamos_personales"                    
 [24] "cpayroll_trx"                             
 [25] "mpayroll"                                 
 [26] "mttarjeta_visa_debitos_automaticos"       
 [27] "ccomisiones_mantenimiento"                
 [28] "mcomisiones_mantenimiento"                
 [29] "ccomisiones_otras"                        
 [30] "mtransferencias_recibidas"                
 [31] "ccallcenter_transacciones"                
 [32] "thomebanking"                             
 [33] "chomebanking_transacciones"               
 [34] "ctrx_quarter"                             
 [35] "Master_status"                            
 [36] "Master_mfinanciacion_limite"              
 [37] "Master_Fvencimiento"                      
 [38] "Master_msaldototal"                       
 [39] "Master_mlimitecompra"                     
 [40] "Master_fultimo_cierre"                    
 [41] "Master_fechaalta"                         
 [42] "Master_mconsumototal"                     
 [43] "Master_cconsumos"                         
 [44] "Master_mpagominimo"                       
 [45] "Visa_status"                              
 [46] "Visa_mfinanciacion_limite"                
 [47] "Visa_Fvencimiento"                        
 [48] "Visa_msaldototal"                         
 [49] "Visa_mlimitecompra"                       
 [50] "Visa_fultimo_cierre"                      
 [51] "Visa_fechaalta"                           
 [52] "Visa_mconsumototal"                       
 [53] "Visa_cconsumos"                           
 [54] "Visa_mpagominimo"                         
 [55] "clase_ternaria"                           
 [56] "kmes"                                     
 [57] "mpayroll_sobre_edad"                      
 [58] "Visa_utilizacion"                         
 [59] "Master_utilizacion"                       
 [60] "Visa_estres_fin"                          
 [61] "Master_estres_fin"                        
 [62] "Visa_pago_minimo_ratio"                   
 [63] "Master_pago_minimo_ratio"                 
 [64] "TC_deuda_total"                           
 [65] "TC_consumo_total"                         
 [66] "TC_trx_total"                             
 [67] "TC_limite_total"                          
 [68] "ratio_deuda_sueldo"                       
 [69] "ratio_liquidez_sueldo"                    
 [70] "ratio_consumo_sueldo"                     
 [71] "Visa_ticket_prom"                         
 [72] "Master_ticket_prom"                       
 [73] "TC_ticket_prom_total"                     
 [74] "ratio_hb_trx"                             
 [75] "ratio_rentabilidad_anual"                 
 [76] "velocidad_productos"                      
 [77] "flag_sobregiro"                           
 [78] "flag_inactivo_deuda"                      
 [79] "flag_doble_tarjeta"                       
 [80] "clase01"                                  
 [

#### 9.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  ni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [34]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 9.3.2 Modelado

#### 9.3.2.1 Training Strategy

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 201901, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 201901, 202105 ]  donde se consideran el 100% de los CONTINUA

In [35]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)


PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [36]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [37]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

In [38]:
if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

# datos de validation (fijos para todas las semillas)
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= FALSE
)
cat("Filas dvalidate:", nrow(dvalidate), "\n")

# meses de final train
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107
)
dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# dfinal_train en formato LightGBM (100% de datos, sin undersampling, reutilizable por todas las semillas)
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= FALSE
)
cat("Filas dfinal_train:", nrow(dfinal_train), "\n")

# dataset del futuro para scoring
PARAM$trainingstrategy$future <- c(202109)
dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]
cat("Filas dfuture:", nrow(dfuture), "\n")


Filas dvalidate: 32938 
Filas dfinal_train: 910853 
Filas dfuture: 33080 


In [39]:
# verificacion de dimensiones
cat("Dimensiones dfinal_train:", nrow(dfinal_train), "x", ncol(dfinal_train), "\n")
cat("Dimensiones dvalidate:", nrow(dvalidate), "x", ncol(dvalidate), "\n")


Dimensiones dfinal_train: 910853 x 421 
Dimensiones dvalidate: 32938 x 421 


####  9.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [40]:
# parametros fijos del LightGBM (plantilla base)
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [41]:
# En x llegan los parametros moviles de LightGBM
# devuelve la AUC en validate del modelo entrenado
# en el parametro x llegan los hiperparametros que se estan optimizando

Estimar_AUC_lightgbm <- function(x, dtrain_local, dval_local, param_fijos_local) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(param_fijos_local, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain_local,
    valids= list(valid = dval_local),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  # hago espacio en la memoria
  niter <- modelo_train$best_iter
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}


seteo del Grid Search

In [42]:
# Espacio de búsqueda de hiperparámetros (idéntico al baseline)
tb_grid_template <- CJ(
  num_leaves= c(64, 128, 256, 384, 512),
  min_data_in_leaf= c(64, 256, 512, 1024, 2048),
  feature_fraction= c(0.5, 0.8)
)


### 9.3.3 Automatización Multi-Semilla y Producción
A continuación se define la función modular que ejecuta el ciclo completo de modelado y producción para cada semilla:
1. Creación de la carpeta `semilla_<semilla>` dentro del directorio del experimento `WF<experimento>`.
2. Undersampling estocástico con la semilla indicada.
3. Grid Search de hiperparámetros (o reutilización según `PARAM$gridsearch_por_semilla`).
4. Final Training del modelo LightGBM con dicha semilla.
5. Exportación de `modelo.txt` e `impo.txt` dentro de la carpeta de la semilla.
6. Scoring de probabilidades en `dfuture` y guardado de `prediccion.txt` dentro de la carpeta de la semilla.
7. Generación y exportación de archivos de corte en `kaggle/` con submit automático.
8. Guardado de metadatos `PARAM.yml` en la carpeta de la semilla.


In [43]:
# ==============================================================================
# Función modular que ejecuta el ciclo completo para una semilla dada
# ==============================================================================
ejecutar_experimento_semilla <- function(vsemilla, mejores_hiperparametros_compartidos = NULL) {

  cat("\n===================================================================\n")
  cat(">>> INICIANDO EJECUCION PARA SEMILLA:", vsemilla, "<<<\n")
  cat("===================================================================\n")

  # 1. Crear carpeta identificatoria de la semilla dentro del experimento
  dir_semilla <- file.path(dir_experimento_base, paste0("semilla_", vsemilla))
  dir.create(dir_semilla, showWarnings = FALSE, recursive = TRUE)
  dir_kaggle <- file.path(dir_semilla, "kaggle")
  dir.create(dir_kaggle, showWarnings = FALSE, recursive = TRUE)

  # Parámetros para esta semilla
  PARAM_sem <- copy(PARAM)
  PARAM_sem$semilla_actual <- vsemilla
  PARAM_sem$lgbm$param_fijos$seed <- vsemilla

  # 2. Undersampling con la semilla actual
  set.seed(vsemilla, kind = "L'Ecuyer-CMRG")
  dataset[, azar := runif(nrow(dataset))]
  dataset[, fold_train := foto_mes %in% PARAM_sem$trainingstrategy$training & 
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") | 
     azar < PARAM_sem$trainingstrategy$training_pct ) ]

  dtrain_sem <- lgb.Dataset(
    data = data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
    label = dataset[fold_train == TRUE, clase01],
    free_raw_data = TRUE
  )

  # 3. Grid Search de Hiperparámetros
  arch_grid <- file.path(dir_semilla, "tb_grid_search_01.txt")
  mejores_hiper <- NULL
  auc_optima <- NULL

  if( file.exists(arch_grid) ) {
    cat("Cargando grid search previo de:", arch_grid, "\n")
    tb_nueva <- fread(arch_grid)
    setorder( tb_nueva, -AUC )
    auc_optima <- tb_nueva[1, AUC]
    mejores_hiper <- as.list(tb_nueva[1])
    mejores_hiper$AUC <- NULL
  } else if( is.null(mejores_hiperparametros_compartidos) || isTRUE(PARAM$gridsearch_por_semilla) ) {
    cat("Ejecutando Grid Search para semilla:", vsemilla, "...\n")
    tb_nueva <- copy(tb_grid_template)

    tb_nueva[, c("AUC", "num_iterations") := Estimar_AUC_lightgbm(
      .SD,
      dtrain_local = dtrain_sem,
      dval_local = dvalidate,
      param_fijos_local = PARAM_sem$lgbm$param_fijos
    ), by = 1:nrow(tb_nueva) ]

    fwrite( tb_nueva, file = arch_grid, sep = "\t" )
    setorder( tb_nueva, -AUC )
    auc_optima <- tb_nueva[1, AUC]
    mejores_hiper <- as.list(tb_nueva[1])
    mejores_hiper$AUC <- NULL
  } else {
    cat("Reutilizando mejores hiperparámetros compartidos para semilla:", vsemilla, "...\n")
    mejores_hiper <- copy(mejores_hiperparametros_compartidos)
    auc_optima <- PARAM$out$lgbm$AUC
  }

  PARAM_sem$out$lgbm$AUC <- auc_optima
  PARAM_sem$out$lgbm$mejores_hiperparametros <- mejores_hiper

  # 4. Final Training
  cat("Entrenando modelo final para semilla:", vsemilla, "...\n")
  fijos <- copy(PARAM_sem$lgbm$param_fijos)
  fijos$num_iterations <- NULL
  fijos$early_stopping_rounds <- NULL

  param_final <- c(fijos, mejores_hiper)
  param_final$seed <- vsemilla

  set.seed(vsemilla, kind = "L'Ecuyer-CMRG")
  final_model <- lgb.train(
    data = dfinal_train,
    param = param_final,
    verbose = -100
  )

  # Grabo modelo en la carpeta de la semilla
  arch_modelo <- file.path(dir_semilla, "modelo.txt")
  lgb.save(final_model, arch_modelo)

  # Importancia de variables en la carpeta de la semilla
  tb_importancia <- as.data.table(lgb.importance(final_model))
  fwrite( tb_importancia, file = file.path(dir_semilla, "impo.txt"), sep = "\t" )

  # 5. Scoring sobre dfuture
  cat("Generando predicciones sobre dfuture para semilla:", vsemilla, "...\n")
  prediccion <- predict(
    final_model,
    data.matrix(dfuture[, campos_buenos, with = FALSE])
  )

  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, prob := prediccion]
  fwrite( tb_prediccion, file = file.path(dir_semilla, "prediccion.txt"), sep = "\t" )

  # 6. Kaggle Competition Submit
  PARAM_sem$kaggle$competencia <- "utn-2026-virtual-jr"
  PARAM_sem$kaggle$cortes <- seq(1800, 2400, by = 100)

  setorder(tb_prediccion, -prob)

  for (envios in PARAM_sem$kaggle$cortes) {
    tb_prediccion[, Predicted := 0L]
    tb_prediccion[1:envios, Predicted := 1L]

    archivo_kaggle <- file.path(dir_kaggle, paste0("KA", PARAM_sem$experimento, "_s", vsemilla, "_", envios, ".csv"))

    fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
      file = archivo_kaggle,
      sep = ","
    )

    comando <- "kaggle competitions submit"
    competencia <- paste("-c", PARAM_sem$kaggle$competencia)
    arch <- paste("-f", archivo_kaggle)
    mensaje <- paste0("-m 'envios=", envios, "  semilla=", vsemilla, "'")
    linea <- paste(comando, competencia, arch, mensaje)

    tryCatch({
      salida <- system(linea, intern = TRUE)
      cat(salida, "\n")
      Sys.sleep(5)
    }, error = function(e) {
      cat("Aviso submit Kaggle:", conditionMessage(e), "\n")
    })
  }

  # 7. Grabo los parametros en la carpeta de la semilla
  if( !require("yaml")) install.packages("yaml")
  require("yaml")
  write_yaml( PARAM_sem, file = file.path(dir_semilla, "PARAM.yml") )

  # Limpieza de memoria
  rm(dtrain_sem, final_model)
  gc(full = TRUE, verbose = FALSE)

  cat(">>> FINALIZADA EXITOSAMENTE SEMILLA:", vsemilla, "<<<\n\n")

  return( list(mejores_hiper = mejores_hiper, AUC = auc_optima) )
}


##### Bucle de Automatización Multi-Semilla


In [44]:
# ==============================================================================
# Bucle de automatización para todas las semillas configuradas
# ==============================================================================
cat("Semillas a ejecutar:", paste(PARAM$semillas, collapse = ", "), "\n")

mejores_hiper_compartidos <- NULL

for( idx in seq_along(PARAM$semillas) ) {
  sem <- PARAM$semillas[idx]

  res <- ejecutar_experimento_semilla(
    vsemilla = sem,
    mejores_hiperparametros_compartidos = mejores_hiper_compartidos
  )

  # Si se optimiza una sola vez, guardamos los mejores hiperparámetros para las siguientes semillas
  if( idx == 1 && !isTRUE(PARAM$gridsearch_por_semilla) ) {
    mejores_hiper_compartidos <- res$mejores_hiper
    PARAM$out$lgbm$AUC <- res$AUC
    PARAM$out$lgbm$mejores_hiperparametros <- res$mejores_hiper
  }
}


Semillas a ejecutar: 100313, 214351, 357913, 701819, 900001 

>>> INICIANDO EJECUCION PARA SEMILLA: 100313 <<<
Ejecutando Grid Search para semilla: 100313 ...


Fri Sep 11 16:34:00 2026  64, 64, 0.5 niter 8 AUC 0.999997521800228

Fri Sep 11 16:34:28 2026  64, 64, 0.8 niter 140 AUC 0.999996563562982

Fri Sep 11 16:36:00 2026  64, 256, 0.5 niter 264 AUC 0.999996695733637

Fri Sep 11 16:37:12 2026  64, 256, 0.8 niter 575 AUC 0.999997488757564

Fri Sep 11 16:38:11 2026  64, 512, 0.5 niter 84 AUC 0.99999735658691

Fri Sep 11 16:39:08 2026  64, 512, 0.8 niter 401 AUC 0.99999735658691

Fri Sep 11 16:39:59 2026  64, 1024, 0.5 niter 51 AUC 0.999997422672237

Fri Sep 11 16:40:32 2026  64, 1024, 0.8 niter 86 AUC 0.999997224416255

Fri Sep 11 16:41:49 2026  64, 2048, 0.5 niter 175 AUC 0.999997620928219

Fri Sep 11 16:42:46 2026  64, 2048, 0.8 niter 166 AUC 0.999997687013546

Fri Sep 11 16:44:40 2026  128, 64, 0.5 niter 338 AUC 0.999997488757564

Fri Sep 11 16:45:29 2026  128, 64, 0.8 niter 306 AUC 0.999997158330928

Fri Sep 11 16:47:22 2026  128, 256, 0.5 niter 329 AUC 0.999997488757564

Fri Sep 11 16:48:12 2026  128, 256, 0.8 niter 286 AUC 0.999998083525

Entrenando modelo final para semilla: 100313 ...
Generando predicciones sobre dfuture para semilla: 100313 ...
92 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
91 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
90 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
89 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
88 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
87 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
86 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 


Loading required package: yaml



>>> FINALIZADA EXITOSAMENTE SEMILLA: 100313 <<<


>>> INICIANDO EJECUCION PARA SEMILLA: 214351 <<<
Reutilizando mejores hiperparámetros compartidos para semilla: 214351 ...
Entrenando modelo final para semilla: 214351 ...
Generando predicciones sobre dfuture para semilla: 214351 ...
85 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
84 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
83 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
82 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
81 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
80 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
79 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
>>> FINALIZADA EXITOSAMENTE SEMILLA: 214351 <<<


>>> INICIANDO EJECUCION PARA SEMILLA: 357913 <<<
Reutilizando mejores hiperparámetros compartidos para semilla: 3

##### Ensamble Multimodelo (Blending)


In [45]:
# ==============================================================================
# Ensamble Multimodelo (Blending) sobre todas las semillas ejecutadas
# ==============================================================================
if( length(PARAM$semillas) > 1 ) {
  cat("\nConstruyendo ensamble de probabilidades para", length(PARAM$semillas), "semillas...\n")

  tb_ensemble <- dfuture[, list(numero_de_cliente)]
  tb_ensemble[, prob_total := 0]
  semillas_validas <- 0

  for( sem in PARAM$semillas ) {
    arch_pred <- file.path(dir_experimento_base, paste0("semilla_", sem), "prediccion.txt")
    if( file.exists(arch_pred) ) {
      tb_sem <- fread(arch_pred)
      tb_ensemble[, paste0("prob_s", sem) := tb_sem$prob]
      tb_ensemble[, prob_total := prob_total + tb_sem$prob]
      semillas_validas <- semillas_validas + 1
    }
  }

  if( semillas_validas > 1 ) {
    tb_ensemble[, prob := prob_total / semillas_validas]
    tb_ensemble[, prob_total := NULL]

    fwrite( tb_ensemble,
      file = file.path(dir_experimento_base, "prediccion_ensemble.txt"),
      sep = "\t"
    )

    dir_kaggle_ens <- file.path(dir_experimento_base, "kaggle_ensemble")
    dir.create(dir_kaggle_ens, showWarnings = FALSE)

    setorder( tb_ensemble, -prob )

    for (envios in PARAM$kaggle$cortes) {
      tb_ensemble[, Predicted := 0L]
      tb_ensemble[1:envios, Predicted := 1L]

      archivo_kaggle_ens <- file.path(dir_kaggle_ens, paste0("KA", PARAM$experimento, "_ensemble_", semillas_validas, "sem_", envios, ".csv"))

      fwrite( tb_ensemble[, list(numero_de_cliente, Predicted)],
        file = archivo_kaggle_ens,
        sep = ","
      )
    }
    cat("Ensamble generado exitosamente en:", dir_kaggle_ens, "\n")
  }
}



Construyendo ensamble de probabilidades para 5 semillas...
Ensamble generado exitosamente en: /content/buckets/b1/exp/WF9111/kaggle_ensemble 


### 9.3.4 Finalización del Workflow


In [46]:
format(Sys.time(), "%a %b %d %X %Y")


[1] "Fri Sep 11 17:58:15 2026"